In [114]:
import numpy as np
import xarray as xr

""" Load data  """

import xarray as xr
from datetime import datetime, timedelta
import numpy as np

days = 3.

expid='iei6'
level=50.

g = xr.open_dataset(f'/hpcperm/pavn/iver/grib/{expid}_w_fc.nc.grb')

ns = 24.*60.*60.*1e9
u = g['u'].sel(step=np.array([days*ns]).astype('timedelta64[ns]')).sel(latitude=slice(50,20)).sel(longitude=slice(0,100)).sel(isobaricInhPa=level)
w = g['w'].sel(step=np.array([days*ns]).astype('timedelta64[ns]')).sel(latitude=slice(50,20)).sel(longitude=slice(0,100)).sel(isobaricInhPa=level)

lat = np.radians(u.latitude.data)
lon = np.radians(u.longitude.data)
Nx = lon.shape[0]
My = lat.shape[0]

# Compute fluxes using FFTs

In [74]:
uhat = np.fft.fft2(u.data)
what = np.fft.fft2(w.data)
""" To remove the mean, we set the k=0,l=0 coefficients to 0 """

uhat[...,0,0] = 0.
what[...,0,0] = 0.


flux_hat = np.conj(what)*uhat/(Nx*My)

# Compute fluxes by removing mean

In [75]:
uprime = u.data - u.mean(dim=['latitude','longitude']).data[...,np.newaxis,np.newaxis]
wprime = w.data - w.mean(dim=['latitude','longitude']).data[...,np.newaxis,np.newaxis]

fluxes = uprime * wprime

In [63]:
total_flux = np.sum(fluxes,axis=(-2,-1)).mean() # sum over space and mean over time

In [64]:
total_flux

np.float32(6.4778023)

In [10]:
total_hat_flux = np.real(np.sum(flux_hat,axis=(-2,-1))).mean() # sum over wavenumbers and mean over time

In [11]:
total_hat_flux

np.float32(6.477802)

# Show that they are equal

In [12]:
np.allclose(total_flux, total_hat_flux)

True

# Compute total wavenumber

In [13]:
radius = 6400000.
dx = radius*np.cos(lat[int(My/2.)])*np.abs(lon[1] - lon[0])
dy = radius*np.abs(lat[1] - lat[0])

k = 2*np.pi*np.fft.fftfreq(Nx,dx)
l = 2*np.pi*np.fft.fftfreq(My,dy)
kk, ll = np.meshgrid(k,l)
ktot = (kk**2 + ll**2)**0.5

In [16]:
ktot_flat = ktot.flatten()
fluxes_flat = flux_hat.mean(axis=(0,1)).flatten()

# Get the indices that would sort ktot in ascending order
sort_indices = np.argsort(ktot_flat)

In [17]:
import metview as mv
import matplotlib.pyplot as plt
%matplotlib ipympl
import matplotlib.pyplot as plt

plt.figure()
plt.plot(ktot_flat[sort_indices], fluxes_flat[sort_indices])

Could not run the Metview executable ('metview'); check that the binaries for Metview (version 5 at least) are installed and are in the PATH.


FileNotFoundError: [Errno 2] No such file or directory: 'metview'

# Note that there are two lines ontop of each other because -ve k and l are repreated because ktot does not know about direction - you could sum the symmetric parts to get the total flux for a total wavenumber